# Airplane Turkish - English Translation - Evaluation

This notebook evaluates the **MIS 48B airplane-domain Turkish - English translation** project.

We fine-tuned `meta-llama/Llama-3.2-1B-Instruct` with LoRA on ~50k synthetic airplane / cabin / boarding
translation pairs. This notebook measures **how good the fine-tuned model is** and **whether fine-tuning
actually helped**, using several complementary metrics.

The evaluation separates two kinds of evidence:

1. **Direct in-domain comparison (the main result).** Every model is run on the *same* held-out airplane
   examples from this project, so the numbers are directly comparable.
2. **External published benchmarks (context only).** Public BLEU / chrF values from model cards and WMT
   papers. Useful background, but *not* directly comparable because they use different datasets.

**Models compared (in-domain):**
- Our fine-tuned Llama-3.2-1B (the project model)
- Base Llama-3.2-1B-Instruct (same model *before* fine-tuning -> shows the fine-tuning gain)
- Helsinki-NLP OPUS-MT (a strong dedicated EN<->TR translation baseline)
- NLLB-200 distilled 600M (a multilingual translation baseline)
- NLLB-200 1.3B (a stronger optional NLLB baseline)
- MADLAD-400 3B MT (a stronger optional multilingual MT baseline)
- M2M100 418M (an additional multilingual translation baseline)
- mBART-50 many-to-many (an additional multilingual translation baseline)

**Metrics computed:** BLEU, chrF++, TER, BERTScore, COMET, ROUGE-L, Token F1, edit similarity, length/repetition diagnostics, paired win-rate comparisons, plus app-behavior checks
(prompt leakage, source-copy, empty output, extra explanation) and latency. Results are grouped by
direction, domain, difficulty, and tone, and saved as CSV + PNG + a Markdown report.

> Default run = all enabled models on a fixed stratified **300-example** sample. Cache/resume is enabled for long runs.
> Set `RUN_FULL_TEST_SET = True` to use the whole held-out split.

In [ ]:
# 1. Runtime setup -- run this cell FIRST in Colab.
# IMPORTANT: after this cell finishes, restart the runtime before running cell 2.
# This avoids NumPy binary-state errors, broken Transformers lazy model-class mappings,
# and mismatched torchvision imports.

# Keep NumPy and pandas compatible with current Colab / Python 3.12.
!pip install -q --force-reinstall --no-cache-dir "numpy==2.0.2" "pandas==2.2.2"

# Pin Transformers to a known-good release for Llama, Marian, M2M100/NLLB, T5/MADLAD, and mBART.
# Newer or partially-upgraded Colab environments can fail with errors such as
# "Could not find LlamaForCausalLM / MarianMTModel / M2M100ForConditionalGeneration".
!pip install -q --force-reinstall --no-cache-dir     "transformers==4.48.3" "tokenizers>=0.21,<0.22"     "accelerate>=0.33,<2" "huggingface_hub>=0.27,<1" "safetensors>=0.4.5"

# Install evaluation dependencies without asking pip to upgrade the whole Colab stack.
!pip install -q --no-cache-dir     "datasets>=2.20" "sacrebleu>=2.4" "bert-score>=0.3.13"     "sentencepiece>=0.2.0" "sacremoses>=0.1.1"     "matplotlib>=3.7,<3.10" "seaborn>=0.13"     "tqdm>=4.66" "tabulate>=0.9" "ftfy>=6.2"

# bitsandbytes is only needed if you set USE_4BIT = True later.
!pip install -q --no-cache-dir "bitsandbytes>=0.43"

# Translation evaluation does not need torchvision. A mismatched Colab torchvision wheel can break
# Transformers imports with: RuntimeError: operator torchvision::nms does not exist.
!pip uninstall -y -q torchvision

# Re-pin after dependency installation in case a transitive dependency tried to move them.
!pip install -q --force-reinstall --no-cache-dir "numpy==2.0.2" "pandas==2.2.2"

print("Setup complete.")
print("Now restart the Colab runtime: Runtime -> Restart runtime, then continue from cell 2.")

In [ ]:
# 2. Imports and global configuration
# If Colab lost packages after a restart, install only the missing runtime dependencies here.

import importlib.util
import sys

_missing = []
_required_packages = {
    "datasets": "datasets>=2.20",
    "huggingface_hub": "huggingface_hub>=0.24",
    "sacrebleu": "sacrebleu>=2.4",
    "bert_score": "bert-score>=0.3.13",
    "sentencepiece": "sentencepiece>=0.2.0",
    "sacremoses": "sacremoses>=0.1.1",
    "transformers": "transformers==4.48.3",
    "accelerate": "accelerate>=0.33",
    "tabulate": "tabulate>=0.9",
    "ftfy": "ftfy>=6.2",
}
for module_name, package_spec in _required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        _missing.append(package_spec)

if _missing:
    print("Installing missing packages:", _missing)
    !{sys.executable} -m pip install -q --no-cache-dir {" ".join(_missing)}
    print("Missing packages installed. If the next import still fails, restart runtime and rerun from this cell.")

from importlib.metadata import PackageNotFoundError, version
try:
    _transformers_version = version("transformers")
except PackageNotFoundError:
    _transformers_version = None
if _transformers_version != "4.48.3":
    print("Transformers version is", _transformers_version, "but this notebook expects 4.48.3.")
    !{sys.executable} -m pip install -q --force-reinstall --no-cache-dir "transformers==4.48.3" "tokenizers>=0.21,<0.22" "accelerate>=0.33,<2" "huggingface_hub>=0.27,<1" "safetensors>=0.4.5"
    raise RuntimeError("Transformers was repaired. Restart the Colab runtime, then rerun from cell 2.")


import gc
from difflib import SequenceMatcher
import json
import os

# This is a text translation notebook. Disable/remove broken torchvision before importing Transformers.
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")
if importlib.util.find_spec("torchvision") is not None:
    try:
        import torchvision  # noqa: F401
    except Exception as exc:
        print("Broken torchvision detected; removing it because this notebook does not need vision models.")
        !{sys.executable} -m pip uninstall -y -q torchvision
        raise RuntimeError(
            "Broken torchvision was removed. Restart the Colab runtime, then rerun from cell 2. "
            "This fixes: operator torchvision::nms does not exist."
        ) from exc

import re
import time
import unicodedata
from datetime import date
from pathlib import Path
from typing import Any

from ftfy import fix_text

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from huggingface_hub import login
from sacrebleu.metrics import BLEU, CHRF, TER
from tqdm.auto import tqdm
try:
    from transformers import (
        AutoTokenizer,
        LlamaForCausalLM,
        MarianMTModel,
        M2M100ForConditionalGeneration,
        MBartForConditionalGeneration,
        T5ForConditionalGeneration,
    )
except Exception as exc:
    raise RuntimeError(
        "Transformers model classes failed to import. Run setup cell 1, restart the Colab runtime, "
        "then continue from cell 2. This repairs the package stack used by Llama, OPUS/Marian, NLLB/M2M100, MADLAD/T5, and mBART."
    ) from exc

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

# Fail early if the Colab package stack is broken. If this raises, rerun setup cell 1,
# restart the runtime, then continue from this cell.
import transformers
from transformers.utils import is_torch_available
if not is_torch_available():
    raise RuntimeError("Transformers does not see PyTorch. Run setup cell 1, restart runtime, then rerun from cell 2.")
_REQUIRED_MODEL_CLASSES = [
    LlamaForCausalLM,
    MarianMTModel,
    M2M100ForConditionalGeneration,
    MBartForConditionalGeneration,
    T5ForConditionalGeneration,
]
print("transformers version:", transformers.__version__)

# ---- Run configuration (toggle these) -------------------------------------
SEED = 42
RUN_FULL_TEST_SET = False     # True -> use the whole held-out split (slower)
EVAL_SAMPLE_SIZE = 300        # used when RUN_FULL_TEST_SET = False

RUN_BASE_LLAMA = True         # base Llama-3.2-1B is GATED -> needs a HF token (see cell 4)
RUN_NLLB = True               # facebook/nllb-200-distilled-600M (public)
RUN_NLLB_1_3B = True          # facebook/nllb-200-1.3B; stronger but slower/larger
RUN_MADLAD400_3B = True       # google/madlad400-3b-mt; stronger but much larger
RUN_M2M100 = True             # facebook/m2m100_418M; extra multilingual baseline
RUN_MBART50 = True            # facebook/mbart-large-50-many-to-many-mmt; extra baseline
RUN_COMET = True              # COMET neural metric; downloads a large model, GPU recommended

USE_4BIT = False              # load the 1B causal models in 4-bit (only for low-memory GPUs)
FORCE_REGENERATE_PREDICTIONS = False   # True -> ignore cached predictions.csv and re-run models
REQUIRE_FINAL_DATASET = True              # fail early if final/50k dataset is missing

MAX_NEW_TOKENS = 80
GENERATION_BATCH_SIZE_SEQ2SEQ = 8
BERTSCORE_BATCH_SIZE = 16
COMET_BATCH_SIZE = 8
N_BOOTSTRAP = 200             # bootstrap resamples for confidence intervals

# ---- Model ids ------------------------------------------------------------
BASE_LLAMA_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
OPUS_EN_TR_MODEL_ID = "Helsinki-NLP/opus-mt-tc-big-en-tr"
OPUS_TR_EN_MODEL_ID = "Helsinki-NLP/opus-mt-tc-big-tr-en"
NLLB_MODEL_ID = "facebook/nllb-200-distilled-600M"
NLLB_1_3B_MODEL_ID = "facebook/nllb-200-1.3B"
MADLAD400_3B_MODEL_ID = "google/madlad400-3b-mt"
M2M100_MODEL_ID = "facebook/m2m100_418M"
MBART50_MODEL_ID = "facebook/mbart-large-50-many-to-many-mmt"
COMET_MODEL_ID = "Unbabel/wmt22-comet-da"

# The system prompt the model was fine-tuned with (kept identical for a fair test).
TRANSLATION_SYSTEM_PROMPT = """You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.

Your task is to translate the user's sentence between Turkish and English.

Rules:
- If the input is Turkish, translate it into natural English.
- If the input is English, translate it into natural Turkish.
- Preserve the meaning, politeness level, urgency, and speaker intent.
- Use simple, clear, practical language suitable for airplane passengers and cabin crew.
- Do not add explanations.
- Do not answer the user's request.
- Do not roleplay.
- Only return the translated sentence.
- For emergency sentences, keep the translation direct and accurate.
- For polite requests, preserve politeness naturally.
- For announcements or crew instructions, use clear formal language.
"""

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("transformers + torch ready.")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# 3. Project paths
# Works in Colab when the MIS48B+ folder is in Google Drive, or uploaded to /content.

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)

CANDIDATE_PROJECT_ROOTS = [
    Path("/content/drive/MyDrive/MIS48B+"),
    Path("/content/MIS48B+"),
    Path.cwd(),
]

PROJECT_ROOT = next(
    (p for p in CANDIDATE_PROJECT_ROOTS if (p / "airplane_translation_dataset").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the MIS48B+ project root. Upload the MIS48B+ folder to /content, "
        "or place it in /content/drive/MyDrive/MIS48B+."
    )

FINAL_DATA_FILE = PROJECT_ROOT / "airplane_translation_dataset" / "final" / "fine_tune_chat.jsonl"
LEGACY_DATA_FILE = PROJECT_ROOT / "airplane_translation_dataset" / "fine_tune_chat.jsonl"

# Prefer the exact final dataset path. If it is missing, search Drive for the
# largest/final fine_tune_chat.jsonl so a differently placed Drive upload still works.
def find_best_fine_tune_file(project_root: Path) -> Path:
    if FINAL_DATA_FILE.exists():
        return FINAL_DATA_FILE

    search_roots = [project_root, Path("/content/drive/MyDrive"), Path("/content")]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for candidate in root.rglob("fine_tune_chat.jsonl"):
                try:
                    size = candidate.stat().st_size
                except OSError:
                    continue
                score = size
                if "final" in [part.lower() for part in candidate.parts]:
                    score += 10**12
                candidates.append((score, size, candidate))
        except OSError:
            continue

    if candidates:
        candidates = sorted(candidates, reverse=True)
        print("Found fine_tune_chat.jsonl candidates:")
        for _, size, candidate in candidates[:10]:
            print(f"  {size/1024/1024:.2f} MB  {candidate}")
        return candidates[0][2]

    return LEGACY_DATA_FILE

DATA_FILE = find_best_fine_tune_file(PROJECT_ROOT)

MERGED_MODEL_DIR = PROJECT_ROOT / "airplane_translation_model" / "merged_final_model"
OUTPUT_DIR = PROJECT_ROOT / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.csv"
METRICS_SUMMARY_PATH = OUTPUT_DIR / "metrics_summary.csv"
GROUPED_METRICS_PATH = OUTPUT_DIR / "grouped_metrics.csv"
LEADERBOARD_PATH = OUTPUT_DIR / "leaderboard.csv"
QUALITATIVE_PATH = OUTPUT_DIR / "qualitative_examples.csv"
INTERNET_BENCHMARKS_PATH = OUTPUT_DIR / "internet_benchmarks.csv"
REPORT_PATH = OUTPUT_DIR / "evaluation_report.md"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("DATA_FILE        :", DATA_FILE)
print("MERGED_MODEL_DIR :", MERGED_MODEL_DIR)
print("OUTPUT_DIR       :", OUTPUT_DIR)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")
if DATA_FILE == LEGACY_DATA_FILE:
    message = (
        "final/fine_tune_chat.jsonl was not found. The notebook would use the smaller legacy dataset, "
        "which is not the real project evaluation file. Upload "
        "MIS48B+/airplane_translation_dataset/final/fine_tune_chat.jsonl."
    )
    if REQUIRE_FINAL_DATASET:
        raise FileNotFoundError(message)
    print("WARNING:", message)

if not MERGED_MODEL_DIR.exists():
    raise FileNotFoundError(f"Missing merged model folder: {MERGED_MODEL_DIR}")

# Transformers needs a standard model.safetensors filename or an index file.
weight_files = sorted(MERGED_MODEL_DIR.glob("*.safetensors"))
has_standard_weight = (MERGED_MODEL_DIR / "model.safetensors").exists()
has_index = any(MERGED_MODEL_DIR.glob("*.index.json"))
if weight_files and not has_standard_weight and not has_index and len(weight_files) == 1:
    target = MERGED_MODEL_DIR / "model.safetensors"
    print(f"Renaming {weight_files[0].name} -> model.safetensors so Transformers can load it.")
    weight_files[0].rename(target)

config_path = MERGED_MODEL_DIR / "config.json"
if not config_path.exists():
    raise FileNotFoundError(f"Missing model config: {config_path}")
with config_path.open("r", encoding="utf-8") as f:
    model_config = json.load(f)

if "model_type" not in model_config:
    print("WARNING: merged model config.json is missing model_type. Repairing it as a Llama config.")
    model_config.setdefault("model_type", "llama")
    model_config.setdefault("architectures", ["LlamaForCausalLM"])
    model_config.setdefault("vocab_size", 128256)
    model_config.setdefault("hidden_size", 2048)
    model_config.setdefault("intermediate_size", 8192)
    model_config.setdefault("num_hidden_layers", 16)
    model_config.setdefault("num_attention_heads", 32)
    model_config.setdefault("num_key_value_heads", 8)
    model_config.setdefault("hidden_act", "silu")
    model_config.setdefault("max_position_embeddings", 131072)
    model_config.setdefault("rope_theta", 500000.0)
    model_config.setdefault("attention_bias", False)
    model_config.setdefault("attention_dropout", 0.0)
    model_config.setdefault("rms_norm_eps", 1e-5)
    model_config.setdefault("tie_word_embeddings", True)
    model_config.setdefault("bos_token_id", 128000)
    model_config.setdefault("eos_token_id", [128001, 128008, 128009])
    with config_path.open("w", encoding="utf-8") as f:
        json.dump(model_config, f, ensure_ascii=False, indent=2)

if model_config.get("model_type") != "llama":
    raise ValueError(f"Expected Llama merged model config, got model_type={model_config.get('model_type')!r}")

print("Merged model config model_type:", model_config.get("model_type"))
# The exported tokenizer_config.json may contain tokenizer_class=TokenizersBackend,
# which is not an importable Hugging Face tokenizer class. Repair it for Colab.
tokenizer_config_path = MERGED_MODEL_DIR / "tokenizer_config.json"
if tokenizer_config_path.exists():
    with tokenizer_config_path.open("r", encoding="utf-8") as f:
        tokenizer_config = json.load(f)
    if tokenizer_config.get("tokenizer_class") == "TokenizersBackend":
        print("WARNING: tokenizer_config.json has tokenizer_class=TokenizersBackend. Repairing to PreTrainedTokenizerFast.")
        tokenizer_config["tokenizer_class"] = "PreTrainedTokenizerFast"
        tokenizer_config.setdefault("eos_token", "<|eot_id|>")
        tokenizer_config.setdefault("bos_token", "<|begin_of_text|>")
        tokenizer_config.setdefault("pad_token", "<|eot_id|>")
        with tokenizer_config_path.open("w", encoding="utf-8") as f:
            json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)
    print("Tokenizer config tokenizer_class:", tokenizer_config.get("tokenizer_class"))


In [ ]:
# 4. Hugging Face authentication (only needed for the GATED base Llama model)
#
# To compare against base Llama-3.2-1B you must:
#   1) Accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
#   2) Create a READ token at https://huggingface.co/settings/tokens
#   3) Add it as a Colab secret named HF_TOKEN (key icon on the left), then run this cell.
#
# If no token is found, the notebook still runs everything else and just skips base Llama.

def get_hf_token() -> str | None:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        for secret_name in ["HF_TOKEN", "HuggingFac-Write", "HUGGINGFACE_TOKEN"]:
            try:
                token = userdata.get(secret_name)
            except Exception:
                token = None
            if token:
                os.environ["HF_TOKEN"] = token
                return token
    except Exception:
        pass
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login OK -- base Llama comparison enabled.")
    except Exception as exc:
        print("HF login attempted but failed:", exc)
else:
    print("No Hugging Face token found.")
    if RUN_BASE_LLAMA:
        print(" -> Base Llama is gated and will be SKIPPED. Public models still run.")
        print(" -> Add a Colab secret named HF_TOKEN (after accepting the license) to enable it.")

In [ ]:
# 5. Load the dataset and recreate the exact held-out split
# (test_size=0.02, seed=42 -- identical to the training notebook).

raw_dataset = load_dataset("json", data_files=str(DATA_FILE), split="train")
split_dataset = raw_dataset.train_test_split(test_size=0.02, seed=SEED)
heldout_dataset = split_dataset["test"]
print(raw_dataset)
print("Held-out examples:", len(heldout_dataset))
if len(raw_dataset) < 10000:
    message = (
        "This dataset has fewer than 10,000 rows. You are probably using the old top-level fine_tune_chat.jsonl. "
        "Expected project dataset: MIS48B+/airplane_translation_dataset/final/fine_tune_chat.jsonl with about 50,256 rows."
    )
    if REQUIRE_FINAL_DATASET:
        raise ValueError(message)
    print("WARNING:", message)

USER_PATTERN = re.compile(r"^Translate to (Turkish|English):\s*(.*)$", re.IGNORECASE | re.DOTALL)

def fix_text_if_needed(value: Any) -> str:
    text = "" if value is None else str(value)
    fixed = fix_text(text)
    # Some Drive/CSV displays contain classic UTF-8-as-Latin-1 mojibake.
    # ftfy catches most cases; this fallback catches stubborn Turkish strings like ÃœÅŸÃ¼dÃ¼m.
    if re.search(r"[ÃÄÅ]", fixed):
        for encoding in ["latin1", "cp1252"]:
            try:
                candidate = fixed.encode(encoding).decode("utf-8")
                candidate = fix_text(candidate)
                if not re.search(r"[ÃÄÅ]", candidate):
                    return candidate
            except UnicodeError:
                pass
    return fixed

def parse_chat_record(record: dict[str, Any], row_id: int) -> dict[str, Any]:
    messages = record["messages"]
    user_text = fix_text_if_needed(messages[1]["content"]).strip()
    assistant_text = fix_text_if_needed(messages[2]["content"]).strip()
    match = USER_PATTERN.match(user_text)
    if not match:
        raise ValueError(f"Unexpected user prompt at row {row_id}: {user_text[:120]}")

    target_language = match.group(1).title()
    source_text = fix_text_if_needed(match.group(2)).strip()
    source_language = "Turkish" if target_language == "English" else "English"
    metadata = record.get("metadata") or {}

    return {
        "row_id": row_id,
        "source_text": source_text,
        "reference_text": assistant_text,
        "source_language": source_language,
        "target_language": target_language,
        "direction": f"{source_language}->{target_language}",
        "domain": metadata.get("domain", "unknown"),
        "scenario_group": metadata.get("scenario_group", "unknown"),
        "difficulty": metadata.get("difficulty", "unknown"),
        "tone": metadata.get("tone", "unknown"),
        "speaker": metadata.get("speaker", "unknown"),
        "listener": metadata.get("listener", "unknown"),
        "user_prompt": user_text,
    }

heldout_records = [parse_chat_record(rec, i) for i, rec in enumerate(heldout_dataset)]
heldout_df = pd.DataFrame(heldout_records)

# Safety: catch UTF-8 mojibake (corrupted Turkish characters) early.
mojibake_pattern = re.compile(r"[ÃÄÅ]")
mojibake_hits = heldout_df[
    heldout_df["source_text"].str.contains(mojibake_pattern, regex=True, na=False)
    | heldout_df["reference_text"].str.contains(mojibake_pattern, regex=True, na=False)
]
if len(mojibake_hits):
    raise ValueError(f"Possible UTF-8 mojibake in {len(mojibake_hits)} rows -- check file encoding.")

print("\nText repair applied with ftfy where needed.")
print("\nDirection distribution:")
print(heldout_df["direction"].value_counts())
print("\nTop domains:")
print(heldout_df["domain"].value_counts().head(10))
heldout_df.head()

In [ ]:
# 6. Build a fixed, reproducible stratified evaluation sample
# Stratify by direction x domain so every group is represented.

def stratified_sample(df: pd.DataFrame, n: int, strata_cols: list[str], seed: int = SEED) -> pd.DataFrame:
    if n >= len(df):
        return df.copy().reset_index(drop=True)

    work = df.copy()
    work["_stratum"] = work[strata_cols].astype(str).agg(" | ".join, axis=1)
    counts = work["_stratum"].value_counts().sort_index()
    raw_alloc = counts / counts.sum() * n
    alloc = np.floor(raw_alloc).astype(int)
    alloc = alloc.clip(lower=1, upper=counts)

    while int(alloc.sum()) > n:
        reducible = alloc[alloc > 1]
        if reducible.empty:
            break
        idx = (raw_alloc.loc[reducible.index] - alloc.loc[reducible.index]).sort_values().index[0]
        alloc.loc[idx] -= 1
    while int(alloc.sum()) < n:
        expandable = alloc[alloc < counts]
        if expandable.empty:
            break
        idx = (raw_alloc.loc[expandable.index] - alloc.loc[expandable.index]).sort_values(ascending=False).index[0]
        alloc.loc[idx] += 1

    parts = []
    for stratum, k in alloc.items():
        group = work[work["_stratum"] == stratum]
        parts.append(group.sample(n=int(k), random_state=seed))
    sampled = pd.concat(parts, ignore_index=True)
    sampled = sampled.sample(frac=1.0, random_state=seed).drop(columns=["_stratum"]).reset_index(drop=True)
    return sampled

if RUN_FULL_TEST_SET:
    eval_df = heldout_df.copy().reset_index(drop=True)
else:
    eval_df = stratified_sample(heldout_df, EVAL_SAMPLE_SIZE, ["direction", "domain"], SEED)

print("Evaluation examples:", len(eval_df))
eval_df.groupby(["direction", "domain"]).size().reset_index(name="n").head(40)

## Metric guide (for the report)

A short, plain-language description of every metric, and why it fits a **Turkish - English airplane
translation** task. Turkish is *morphologically rich* (one word can carry many suffixes), so character-
and meaning-level metrics matter more than plain word overlap.

| Metric | What it measures | Range | Good? | Why we use it |
|---|---|---|---|---|
| **BLEU** | word n-gram overlap with the reference | 0-100 | higher | Standard MT metric; easy to compare with literature. |
| **chrF++** | character + word n-gram F-score | 0-100 | higher | **Primary metric** here; robust to Turkish suffixes and word order. |
| **TER** | edit operations needed to fix the output | 0-100+ | lower | Intuitive "how much post-editing" measure. |
| **BERTScore** | semantic similarity via multilingual BERT embeddings | 0-1 | higher | Rewards correct meaning even when wording differs. |
| **COMET** *(optional)* | learned neural quality estimate | ~0-1 | higher | Strong correlation with human translation judgement. |
| **ROUGE-L** | longest common subsequence overlap | 0-1 | higher | Captures phrase/order similarity without requiring exact n-grams. |
| **Token F1** | overlap of normalized tokens | 0-1 | higher | Simple interpretable precision/recall style metric. |
| **Edit similarity** | normalized character-level similarity | 0-1 | higher | Easy-to-explain closeness score, useful for slides. |
| **Length ratio** | output length/reference length | near 1 | closer to 1 | Detects outputs that are too short/long. |
| **Repetition rate** | repeated token overuse | 0-1 | lower | Detects degenerate generation loops. |

**App-behavior checks** (does the model behave like a translator, not a chatbot?):
- **source-copy rate** - output just echoes the input (no translation). Lower is better.
- **prompt-leakage rate** - output contains prompt/role words ("translate to", "assistant"). Lower is better.
- **empty-output rate** - model produced nothing. Lower is better.
- **extra-explanation rate** - model added explanations / extra lines instead of only the translation. Lower is better.

**Model comparison design:** the fair comparison is still the in-domain test set, because every enabled
model translates the exact same examples. External internet benchmarks stay separate because they use
different datasets, tokenization, and metric settings.

In [ ]:
# 7. Translation helpers

def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def model_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

def quantization_config():
    if not USE_4BIT or not torch.cuda.is_available() or BitsAndBytesConfig is None:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_dtype(),
        bnb_4bit_use_double_quant=True,
    )

def build_chat_prompt(tokenizer, source_text: str, target_language: str) -> str:
    messages = [
        {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
        {"role": "user", "content": f"Translate to {target_language}: {source_text}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def translate_causal_batch(model, tokenizer, batch: pd.DataFrame) -> list[str]:
    predictions = []
    for row in tqdm(list(batch.itertuples(index=False)), desc="causal", leave=False):
        prompt = build_chat_prompt(tokenizer, row.source_text, row.target_language)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0][inputs["input_ids"].shape[-1]:]
        predictions.append(tokenizer.decode(generated, skip_special_tokens=True).strip())
    return predictions

def translate_opus_batch(model, tokenizer, texts: list[str], device) -> list[str]:
    outputs = []
    for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
        chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
        inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=4)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return [t.strip() for t in outputs]

def translate_nllb_batch(model, tokenizer, batch: pd.DataFrame, device) -> list[str]:
    outputs = []
    lang_codes = {"English": "eng_Latn", "Turkish": "tur_Latn"}
    for _, direction_batch in batch.groupby(["source_language", "target_language"], sort=False):
        src_lang = direction_batch.iloc[0]["source_language"]
        tgt_lang = direction_batch.iloc[0]["target_language"]
        tokenizer.src_lang = lang_codes[src_lang]
        forced_bos = tokenizer.convert_tokens_to_ids(lang_codes[tgt_lang])
        texts = direction_batch["source_text"].tolist()
        dir_out = []
        for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
            chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            with torch.no_grad():
                generated = model.generate(
                    **inputs, forced_bos_token_id=forced_bos,
                    max_new_tokens=MAX_NEW_TOKENS, num_beams=4,
                )
            dir_out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        outputs.extend(zip(direction_batch.index.tolist(), [t.strip() for t in dir_out]))
    return [p for _, p in sorted(outputs, key=lambda item: item[0])]

def translate_m2m100_batch(model, tokenizer, batch: pd.DataFrame, device) -> list[str]:
    outputs = []
    lang_codes = {"English": "en", "Turkish": "tr"}
    for _, direction_batch in batch.groupby(["source_language", "target_language"], sort=False):
        src_lang = direction_batch.iloc[0]["source_language"]
        tgt_lang = direction_batch.iloc[0]["target_language"]
        tokenizer.src_lang = lang_codes[src_lang]
        forced_bos = tokenizer.get_lang_id(lang_codes[tgt_lang])
        texts = direction_batch["source_text"].tolist()
        dir_out = []
        for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
            chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            with torch.no_grad():
                generated = model.generate(
                    **inputs, forced_bos_token_id=forced_bos,
                    max_new_tokens=MAX_NEW_TOKENS, num_beams=4,
                )
            dir_out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        outputs.extend(zip(direction_batch.index.tolist(), [t.strip() for t in dir_out]))
    return [p for _, p in sorted(outputs, key=lambda item: item[0])]

def translate_mbart50_batch(model, tokenizer, batch: pd.DataFrame, device) -> list[str]:
    outputs = []
    lang_codes = {"English": "en_XX", "Turkish": "tr_TR"}
    for _, direction_batch in batch.groupby(["source_language", "target_language"], sort=False):
        src_lang = direction_batch.iloc[0]["source_language"]
        tgt_lang = direction_batch.iloc[0]["target_language"]
        tokenizer.src_lang = lang_codes[src_lang]
        forced_bos = tokenizer.lang_code_to_id[lang_codes[tgt_lang]]
        texts = direction_batch["source_text"].tolist()
        dir_out = []
        for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
            chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            with torch.no_grad():
                generated = model.generate(
                    **inputs, forced_bos_token_id=forced_bos,
                    max_new_tokens=MAX_NEW_TOKENS, num_beams=4,
                )
            dir_out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        outputs.extend(zip(direction_batch.index.tolist(), [t.strip() for t in dir_out]))
    return [p for _, p in sorted(outputs, key=lambda item: item[0])]

def model_input_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def translate_madlad400_batch(model, tokenizer, batch: pd.DataFrame) -> list[str]:
    outputs = []
    target_prefix = {"English": "<2en>", "Turkish": "<2tr>"}
    input_device = model_input_device(model)
    madlad_batch_size = max(1, GENERATION_BATCH_SIZE_SEQ2SEQ // 2)
    for _, direction_batch in batch.groupby(["source_language", "target_language"], sort=False):
        tgt_lang = direction_batch.iloc[0]["target_language"]
        texts = [f"{target_prefix[tgt_lang]} {text}" for text in direction_batch["source_text"].tolist()]
        dir_out = []
        for start in range(0, len(texts), madlad_batch_size):
            chunk = texts[start:start + madlad_batch_size]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(input_device)
            with torch.no_grad():
                generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=4)
            dir_out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        outputs.extend(zip(direction_batch.index.tolist(), [t.strip() for t in dir_out]))
    return [p for _, p in sorted(outputs, key=lambda item: item[0])]

In [ ]:
# 8. Model runners (each model is loaded once, run, then unloaded; failures are tolerated)

MODEL_SPECS = [
    {"model_name": "Fine-tuned Llama 3.2 1B", "model_key": "fine_tuned_llama", "kind": "causal", "path": str(MERGED_MODEL_DIR)},
    {"model_name": "Base Llama 3.2 1B",       "model_key": "base_llama",       "kind": "causal", "path": BASE_LLAMA_MODEL_ID},
    {"model_name": "OPUS-MT tc-big",          "model_key": "opus_mt_tc_big",   "kind": "opus_pair"},
    {"model_name": "NLLB-200 distilled 600M", "model_key": "nllb_200_distilled_600m", "kind": "nllb", "path": NLLB_MODEL_ID},
    {"model_name": "NLLB-200 1.3B",          "model_key": "nllb_200_1_3b",    "kind": "nllb", "path": NLLB_1_3B_MODEL_ID},
    {"model_name": "MADLAD-400 3B MT",       "model_key": "madlad400_3b_mt", "kind": "madlad400", "path": MADLAD400_3B_MODEL_ID},
    {"model_name": "M2M100 418M",             "model_key": "m2m100_418m",      "kind": "m2m100", "path": M2M100_MODEL_ID},
    {"model_name": "mBART-50 many-to-many",   "model_key": "mbart50_mmt",      "kind": "mbart50", "path": MBART50_MODEL_ID},
]

MODEL_TOGGLES = {
    "base_llama": RUN_BASE_LLAMA,
    "nllb_200_distilled_600m": RUN_NLLB,
    "nllb_200_1_3b": RUN_NLLB_1_3B,
    "madlad400_3b_mt": RUN_MADLAD400_3B,
    "m2m100_418m": RUN_M2M100,
    "mbart50_mmt": RUN_MBART50,
}

def model_enabled(spec: dict[str, Any]) -> bool:
    return MODEL_TOGGLES.get(spec["model_key"], True)

def _load_causal(path):
    try:
        tok = AutoTokenizer.from_pretrained(path, use_fast=True, trust_remote_code=True, token=HF_TOKEN)
    except Exception as exc:
        # The local merged export can have tokenizer_class=TokenizersBackend.
        # The model was fine-tuned from Llama 3.2, so the base tokenizer is the correct fallback.
        if str(path) != BASE_LLAMA_MODEL_ID:
            print(f"Local tokenizer load failed ({exc}). Falling back to base Llama tokenizer.")
            tok = AutoTokenizer.from_pretrained(BASE_LLAMA_MODEL_ID, use_fast=True, trust_remote_code=True, token=HF_TOKEN)
        else:
            raise
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = LlamaForCausalLM.from_pretrained(
        path,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=model_dtype(),
        quantization_config=quantization_config(),
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    model.eval()
    return model, tok

def _load_seq2seq(model_id: str, device, model_cls, device_map_auto: bool = False):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    load_kwargs = {"token": HF_TOKEN}
    if torch.cuda.is_available():
        load_kwargs["torch_dtype"] = model_dtype()
        if device_map_auto:
            load_kwargs["device_map"] = "auto"
    model = model_cls.from_pretrained(model_id, **load_kwargs)
    if not device_map_auto:
        model = model.to(device)
    model.eval()
    return model, tok

def run_model_on_dataframe(spec: dict[str, Any], df: pd.DataFrame) -> pd.DataFrame:
    model_name, kind = spec["model_name"], spec["kind"]
    print(f"\n=== Running {model_name} on {len(df)} examples ===")
    start = time.perf_counter()

    if kind == "causal":
        if spec["path"] == BASE_LLAMA_MODEL_ID and not HF_TOKEN:
            raise RuntimeError("Base Llama is gated -- no HF token found. Skipping (see cell 4).")
        model, tok = _load_causal(spec["path"])
        predictions = translate_causal_batch(model, tok, df)
        del model, tok
        cleanup_memory()

    elif kind == "opus_pair":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        collected = []
        for src_lang, tgt_lang, model_id in [
            ("English", "Turkish", OPUS_EN_TR_MODEL_ID),
            ("Turkish", "English", OPUS_TR_EN_MODEL_ID),
        ]:
            dir_df = df[(df["source_language"] == src_lang) & (df["target_language"] == tgt_lang)]
            if dir_df.empty:
                continue
            model, tok = _load_seq2seq(model_id, device, MarianMTModel)
            preds = translate_opus_batch(model, tok, dir_df["source_text"].tolist(), device)
            collected.extend(zip(dir_df.index.tolist(), preds))
            del model, tok
            cleanup_memory()
        predictions = [p for _, p in sorted(collected, key=lambda item: item[0])]

    elif kind == "nllb":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, tok = _load_seq2seq(spec["path"], device, M2M100ForConditionalGeneration)
        predictions = translate_nllb_batch(model, tok, df, device)
        del model, tok
        cleanup_memory()

    elif kind == "m2m100":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, tok = _load_seq2seq(spec["path"], device, M2M100ForConditionalGeneration)
        predictions = translate_m2m100_batch(model, tok, df, device)
        del model, tok
        cleanup_memory()

    elif kind == "mbart50":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, tok = _load_seq2seq(spec["path"], device, MBartForConditionalGeneration)
        predictions = translate_mbart50_batch(model, tok, df, device)
        del model, tok
        cleanup_memory()

    elif kind == "madlad400":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, tok = _load_seq2seq(spec["path"], device, T5ForConditionalGeneration, device_map_auto=torch.cuda.is_available())
        predictions = translate_madlad400_batch(model, tok, df)
        del model, tok
        cleanup_memory()

    else:
        raise ValueError(f"Unknown model kind: {kind}")

    elapsed = time.perf_counter() - start
    if len(predictions) != len(df):
        raise ValueError(f"{model_name} produced {len(predictions)} preds for {len(df)} rows")

    out = df.copy()
    out["model_key"] = spec["model_key"]
    out["model_name"] = model_name
    out["prediction_text"] = predictions
    out["total_model_seconds"] = elapsed
    out["latency_seconds"] = elapsed / max(len(df), 1)
    print(f"    done in {elapsed:.1f}s  ({elapsed/max(len(df),1):.2f}s/example)")
    return out

def run_models(df: pd.DataFrame, specs: list[dict[str, Any]]) -> tuple[list[pd.DataFrame], dict[str, str]]:
    results, status = [], {}
    for spec in specs:
        try:
            res = run_model_on_dataframe(spec, df)
            display(res[["model_name", "direction", "source_text", "reference_text", "prediction_text"]].head(5))
            results.append(res)
            status[spec["model_name"]] = "ok"
        except Exception as exc:
            print(f"!! {spec['model_name']} SKIPPED: {exc}")
            status[spec["model_name"]] = f"skipped: {exc}"
        cleanup_memory()
    return results, status

def cache_part_is_valid(cached_df: pd.DataFrame, model_key: str, expected_row_ids: set[int]) -> bool:
    required_cols = {"row_id", "model_key", "model_name", "prediction_text"}
    if not required_cols.issubset(cached_df.columns):
        return False
    part = cached_df[cached_df["model_key"] == model_key]
    if len(part) != len(expected_row_ids):
        return False
    if set(part["row_id"].tolist()) != expected_row_ids:
        return False
    if part["prediction_text"].isna().any():
        return False
    return True

enabled_model_specs = [spec for spec in MODEL_SPECS if model_enabled(spec)]
expected_model_keys = {spec["model_key"] for spec in enabled_model_specs}
expected_row_ids = set(eval_df["row_id"].tolist())
model_order = {spec["model_key"]: i for i, spec in enumerate(enabled_model_specs)}

cached_parts = []
specs_to_run = enabled_model_specs.copy()
cache_status = {}

if PREDICTIONS_PATH.exists() and not FORCE_REGENERATE_PREDICTIONS:
    cached_df = pd.read_csv(PREDICTIONS_PATH)
    specs_to_run = []
    for spec in enabled_model_specs:
        key = spec["model_key"]
        if cache_part_is_valid(cached_df, key, expected_row_ids):
            part = cached_df[cached_df["model_key"] == key].copy()
            cached_parts.append(part)
            cache_status[spec["model_name"]] = "cached"
        else:
            specs_to_run.append(spec)
            cache_status[spec["model_name"]] = "needs run"
    ignored_keys = sorted(set(cached_df.get("model_key", pd.Series(dtype=str)).dropna().unique()) - expected_model_keys)
    if ignored_keys:
        print("Ignoring cached predictions for disabled/old model keys:", ignored_keys)
    print("Prediction cache status:")
    for spec in enabled_model_specs:
        print(f"  {spec['model_name']:28s}: {cache_status[spec['model_name']]}")
elif FORCE_REGENERATE_PREDICTIONS and PREDICTIONS_PATH.exists():
    print("FORCE_REGENERATE_PREDICTIONS=True, ignoring existing predictions.csv.")

new_parts, run_status = run_models(eval_df, specs_to_run) if specs_to_run else ([], {})

print("\nRun status:")
for spec in enabled_model_specs:
    name = spec["model_name"]
    if name in run_status:
        value = run_status[name]
    elif any(part["model_key"].iloc[0] == spec["model_key"] for part in cached_parts):
        value = "cached"
    else:
        value = "not available"
    print(f"  {name:28s}: {value}")

all_parts = cached_parts + new_parts
if not all_parts:
    raise RuntimeError("No model produced predictions and no valid cache was available -- check setup, HF token, and toggles.")

predictions_df = pd.concat(all_parts, ignore_index=True)
predictions_df = predictions_df[predictions_df["model_key"].isin(expected_model_keys)].copy()
predictions_df["_model_order"] = predictions_df["model_key"].map(model_order)
predictions_df = predictions_df.sort_values(["_model_order", "row_id"]).drop(columns=["_model_order"]).reset_index(drop=True)
predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

MODEL_RUN_STATUS_PATH = OUTPUT_DIR / "model_run_status.csv"
model_status_rows = []
for spec in enabled_model_specs:
    key = spec["model_key"]
    count = int((predictions_df["model_key"] == key).sum())
    model_status_rows.append({
        "model_key": key,
        "model_name": spec["model_name"],
        "enabled": True,
        "expected_rows": len(eval_df),
        "prediction_rows": count,
        "complete": count == len(eval_df),
        "status": run_status.get(spec["model_name"], cache_status.get(spec["model_name"], "not available")),
    })
pd.DataFrame(model_status_rows).to_csv(MODEL_RUN_STATUS_PATH, index=False, encoding="utf-8-sig")
print("Saved predictions to", PREDICTIONS_PATH)
print("Saved model run status to", MODEL_RUN_STATUS_PATH)

predictions_df.head()

In [ ]:
# 9. Behavior columns + corpus/per-example metric helpers

PUNCT_PATTERN = re.compile(r"[^\w\s]", flags=re.UNICODE)
PROMPT_LEAK_PATTERN = re.compile(r"translate to|turkish:|english:|system|assistant|user", re.IGNORECASE)
EXPLANATION_PATTERN = re.compile("(^|\\s)(here is|translation|translated|means|ceviri|tercume)(\\s|$)", re.IGNORECASE)

bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF(word_order=2)   # word_order=2 -> chrF++
ter_metric = TER()

def normalize_for_match(text: Any) -> str:
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", text).strip().casefold()
    text = PUNCT_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalized_tokens(text: Any) -> list[str]:
    return normalize_for_match(text).split()

def token_f1(prediction: Any, reference: Any) -> float:
    pred_tokens = normalized_tokens(prediction)
    ref_tokens = normalized_tokens(reference)
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    ref_counts = {}
    for tok in ref_tokens:
        ref_counts[tok] = ref_counts.get(tok, 0) + 1
    overlap = 0
    for tok in pred_tokens:
        if ref_counts.get(tok, 0) > 0:
            overlap += 1
            ref_counts[tok] -= 1
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def rouge_l_f1(prediction: Any, reference: Any) -> float:
    pred = normalized_tokens(prediction)
    ref = normalized_tokens(reference)
    if not pred and not ref:
        return 1.0
    if not pred or not ref:
        return 0.0
    # Dynamic-programming LCS over normalized tokens.
    prev = [0] * (len(ref) + 1)
    for p_tok in pred:
        cur = [0]
        for j, r_tok in enumerate(ref, start=1):
            cur.append(prev[j - 1] + 1 if p_tok == r_tok else max(prev[j], cur[-1]))
        prev = cur
    lcs = prev[-1]
    precision = lcs / len(pred)
    recall = lcs / len(ref)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def edit_similarity(prediction: Any, reference: Any) -> float:
    return SequenceMatcher(None, normalize_for_match(prediction), normalize_for_match(reference)).ratio()

def length_ratio(prediction: Any, reference: Any) -> float:
    pred_len = max(len(normalized_tokens(prediction)), 0)
    ref_len = len(normalized_tokens(reference))
    return np.nan if ref_len == 0 else pred_len / ref_len

def repetition_rate(text: Any) -> float:
    toks = normalized_tokens(text)
    if len(toks) < 2:
        return 0.0
    repeated = sum(1 for i in range(1, len(toks)) if toks[i] == toks[i - 1])
    return repeated / (len(toks) - 1)

def sentence_chrfpp(prediction: Any, reference: Any) -> float:
    return chrf_metric.sentence_score(str(prediction or ""), [str(reference or "")]).score

def sentence_bleu(prediction: Any, reference: Any) -> float:
    return bleu_metric.sentence_score(str(prediction or ""), [str(reference or "")]).score

def add_behavior_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["prediction_text", "reference_text", "source_text"]:
        out[col] = out[col].fillna("").astype(str)
    out["normalized_prediction"] = out["prediction_text"].map(normalize_for_match)
    out["normalized_reference"] = out["reference_text"].map(normalize_for_match)
    out["normalized_source"] = out["source_text"].map(normalize_for_match)
    out["exact_match"] = out["prediction_text"].str.strip() == out["reference_text"].str.strip()
    out["normalized_exact_match"] = out["normalized_prediction"] == out["normalized_reference"]
    out["empty_output"] = out["prediction_text"].str.strip().eq("")
    out["source_copy"] = out["normalized_prediction"] == out["normalized_source"]
    out["prompt_leakage"] = out["prediction_text"].str.contains(PROMPT_LEAK_PATTERN, regex=True, na=False)
    out["extra_explanation"] = (
        out["prediction_text"].str.contains(EXPLANATION_PATTERN, regex=True, na=False)
        | out["prediction_text"].str.contains("\n", regex=True, na=False)
    )
    out["token_f1"] = [token_f1(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    out["rouge_l_f1"] = [rouge_l_f1(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    out["edit_similarity"] = [edit_similarity(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    out["length_ratio"] = [length_ratio(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    out["abs_length_ratio_error"] = (out["length_ratio"] - 1.0).abs()
    out["repetition_rate"] = out["prediction_text"].map(repetition_rate)
    out["sentence_chrfpp"] = [sentence_chrfpp(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    out["sentence_bleu"] = [sentence_bleu(p, r) for p, r in zip(out["prediction_text"], out["reference_text"])]
    return out

scored_predictions_df = add_behavior_columns(predictions_df)

def corpus_mt_metrics(group: pd.DataFrame) -> dict[str, float]:
    hyps = group["prediction_text"].fillna("").astype(str).tolist()
    refs = group["reference_text"].fillna("").astype(str).tolist()
    if not hyps:
        return {"bleu": np.nan, "chrfpp": np.nan, "ter": np.nan}
    return {
        "bleu": bleu_metric.corpus_score(hyps, [refs]).score,
        "chrfpp": chrf_metric.corpus_score(hyps, [refs]).score,
        "ter": ter_metric.corpus_score(hyps, [refs]).score,
    }

def aggregate_metrics(group: pd.DataFrame) -> dict[str, Any]:
    m = corpus_mt_metrics(group)
    m.update({
        "n": len(group),
        "exact_match_rate": group["exact_match"].mean(),
        "normalized_exact_match_rate": group["normalized_exact_match"].mean(),
        "empty_output_rate": group["empty_output"].mean(),
        "source_copy_rate": group["source_copy"].mean(),
        "prompt_leakage_rate": group["prompt_leakage"].mean(),
        "extra_explanation_rate": group["extra_explanation"].mean(),
        "avg_token_f1": group["token_f1"].mean(),
        "avg_rouge_l_f1": group["rouge_l_f1"].mean(),
        "avg_edit_similarity": group["edit_similarity"].mean(),
        "avg_length_ratio": group["length_ratio"].mean(),
        "avg_abs_length_ratio_error": group["abs_length_ratio_error"].mean(),
        "avg_repetition_rate": group["repetition_rate"].mean(),
        "avg_sentence_chrfpp": group["sentence_chrfpp"].mean(),
        "avg_sentence_bleu": group["sentence_bleu"].mean(),
        "avg_latency_seconds": group["latency_seconds"].mean() if "latency_seconds" in group.columns else np.nan,
    })
    for opt in ["bertscore_f1", "comet"]:
        if opt in group.columns:
            m[opt] = group[opt].mean()
    return m

scored_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
print("Behavior columns and additional metrics added. Metric helpers ready.")

In [ ]:
# 10. BERTScore (multilingual -- scores EN and TR outputs in one pass)

from bert_score import score as bert_score

for col in ["bertscore_precision", "bertscore_recall", "bertscore_f1"]:
    if col not in scored_predictions_df.columns:
        scored_predictions_df[col] = np.nan

needs_bertscore = scored_predictions_df["bertscore_f1"].isna()
if needs_bertscore.any():
    try:
        missing_df = scored_predictions_df[needs_bertscore]
        print(f"Computing BERTScore for {len(missing_df)} missing rows.")
        for model_key, group in tqdm(missing_df.groupby("model_key"), desc="BERTScore"):
            P, R, F1 = bert_score(
                group["prediction_text"].fillna("").astype(str).tolist(),
                group["reference_text"].fillna("").astype(str).tolist(),
                model_type="bert-base-multilingual-cased",
                batch_size=BERTSCORE_BATCH_SIZE,
                verbose=False,
                rescale_with_baseline=False,
                device="cuda" if torch.cuda.is_available() else "cpu",
            )
            idx = group.index
            scored_predictions_df.loc[idx, "bertscore_precision"] = P.cpu().numpy()
            scored_predictions_df.loc[idx, "bertscore_recall"] = R.cpu().numpy()
            scored_predictions_df.loc[idx, "bertscore_f1"] = F1.cpu().numpy()
        print("BERTScore done.")
    except Exception as exc:
        print("BERTScore failed (continuing without it):", exc)
else:
    print("BERTScore already present for all rows; using cached BERTScore columns.")

scored_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
scored_predictions_df[["model_name", "direction", "bertscore_f1", "prediction_text", "reference_text"]].head()

In [ ]:
# 11. COMET neural MT metric
# Reference-based COMET uses source + model output + reference. It is enabled by RUN_COMET = True in cell 2.
# It downloads a large model, so GPU runtime is strongly recommended.

COMET_PATH = OUTPUT_DIR / "comet_scores.csv"

if RUN_COMET:
    try:
        import importlib.util
        import sys
        if importlib.util.find_spec("comet") is None:
            print("Installing unbabel-comet for COMET scoring...")
            !{sys.executable} -m pip install -q --no-cache-dir "unbabel-comet>=2.2.7" "setuptools<81"
            print("If Colab prints dependency resolver warnings here but COMET finishes, those warnings are usually from unrelated preinstalled Colab packages.")
            print("COMET package installed. If import still fails, restart runtime and rerun from this cell.")

        from comet import download_model, load_from_checkpoint

        if "comet" not in scored_predictions_df.columns:
            scored_predictions_df["comet"] = np.nan

        needs_comet_mask = scored_predictions_df["comet"].isna()
        if needs_comet_mask.any():
            missing_df = scored_predictions_df[needs_comet_mask]
            print(f"Computing COMET for {len(missing_df)} missing rows.")
            print(f"Loading COMET model: {COMET_MODEL_ID}")
            comet_checkpoint = download_model(COMET_MODEL_ID)
            comet_model = load_from_checkpoint(comet_checkpoint)

            for model_key, group in tqdm(missing_df.groupby("model_key"), desc="COMET"):
                comet_data = [
                    {"src": str(src), "mt": str(mt), "ref": str(ref)}
                    for src, mt, ref in zip(
                        group["source_text"].fillna(""),
                        group["prediction_text"].fillna(""),
                        group["reference_text"].fillna(""),
                    )
                ]
                try:
                    pred = comet_model.predict(
                        comet_data,
                        batch_size=COMET_BATCH_SIZE,
                        gpus=1 if torch.cuda.is_available() else 0,
                        progress_bar=True,
                    )
                except TypeError:
                    # Compatibility fallback for COMET/Lightning versions that renamed gpus.
                    pred = comet_model.predict(
                        comet_data,
                        batch_size=COMET_BATCH_SIZE,
                        devices=1 if torch.cuda.is_available() else None,
                        accelerator="gpu" if torch.cuda.is_available() else "cpu",
                        progress_bar=True,
                    )
                if isinstance(pred, dict):
                    scores = pred.get("scores")
                else:
                    scores = getattr(pred, "scores", None)
                if scores is None:
                    raise ValueError(f"Unexpected COMET output type: {type(pred)}")
                scored_predictions_df.loc[group.index, "comet"] = np.asarray(scores, dtype=float)

            del comet_model
            cleanup_memory()
            scored_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
            scored_predictions_df[["row_id", "model_key", "model_name", "direction", "comet"]].to_csv(
                COMET_PATH,
                index=False,
                encoding="utf-8-sig",
            )
            print("COMET done. Saved:", COMET_PATH)
        else:
            print("COMET already present for all rows; using cached COMET scores.")
            scored_predictions_df[["row_id", "model_key", "model_name", "direction", "comet"]].to_csv(
                COMET_PATH,
                index=False,
                encoding="utf-8-sig",
            )

    except Exception as exc:
        raise RuntimeError(
            "COMET was enabled but failed. Recommended fix in Colab: run setup cell 1, restart runtime, "
            "then rerun from cell 2. If the COMET model requires Hugging Face access, ensure HF_TOKEN is set. "
            f"Original error: {exc}"
        )
else:
    print("COMET disabled because RUN_COMET = False.")

In [ ]:
# 12. Aggregate metrics: overall + grouped breakdowns

summary_rows = []
for (model_key, model_name), group in scored_predictions_df.groupby(["model_key", "model_name"], sort=False):
    row = {"model_key": model_key, "model_name": model_name, "group_type": "overall", "group_value": "overall"}
    row.update(aggregate_metrics(group))
    summary_rows.append(row)

metrics_summary_df = pd.DataFrame(summary_rows).sort_values("chrfpp", ascending=False).reset_index(drop=True)
metrics_summary_df.to_csv(METRICS_SUMMARY_PATH, index=False, encoding="utf-8-sig")

grouped_rows = []
for group_col in ["direction", "domain", "difficulty", "tone"]:
    for (mk, mn, gv), group in scored_predictions_df.groupby(["model_key", "model_name", group_col], sort=False):
        row = {"model_key": mk, "model_name": mn, "group_type": group_col, "group_value": gv}
        row.update(aggregate_metrics(group))
        grouped_rows.append(row)
grouped_metrics_df = pd.DataFrame(grouped_rows).sort_values(
    ["group_type", "group_value", "chrfpp"], ascending=[True, True, False]
).reset_index(drop=True)
grouped_metrics_df.to_csv(GROUPED_METRICS_PATH, index=False, encoding="utf-8-sig")

print("Saved:", METRICS_SUMMARY_PATH.name, "and", GROUPED_METRICS_PATH.name)
metrics_summary_df

In [ ]:
# 13. Leaderboard + paired model comparison

PAIRED_MODEL_COMPARISON_PATH = OUTPUT_DIR / "paired_model_comparison.csv"

def bootstrap_corpus_ci(group: pd.DataFrame, metric: str = "chrfpp",
                        n_boot: int = N_BOOTSTRAP, seed: int = SEED):
    hyps = group["prediction_text"].fillna("").astype(str).tolist()
    refs = group["reference_text"].fillna("").astype(str).tolist()
    n = len(hyps)
    if n < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    scorer = chrf_metric if metric == "chrfpp" else bleu_metric
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        h = [hyps[i] for i in idx]
        r = [refs[i] for i in idx]
        scores.append(scorer.corpus_score(h, [r]).score)
    return (float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5)))

lb_rows = []
for _, row in metrics_summary_df.iterrows():
    group = scored_predictions_df[scored_predictions_df["model_key"] == row["model_key"]]
    lo, hi = bootstrap_corpus_ci(group, "chrfpp")
    lb_rows.append({
        "model_name": row["model_name"],
        "n": int(row["n"]),
        "BLEU": round(row["bleu"], 2),
        "chrF++": round(row["chrfpp"], 2),
        "chrF++_95CI": f"[{lo:.1f}, {hi:.1f}]",
        "TER": round(row["ter"], 2),
        "BERTScore_F1": round(row["bertscore_f1"], 4) if "bertscore_f1" in row and pd.notna(row["bertscore_f1"]) else None,
        "COMET": round(row["comet"], 4) if "comet" in row and pd.notna(row["comet"]) else None,
        "ROUGE_L_F1": round(row["avg_rouge_l_f1"], 4),
        "Token_F1": round(row["avg_token_f1"], 4),
        "Edit_similarity": round(row["avg_edit_similarity"], 4),
        "Length_ratio": round(row["avg_length_ratio"], 3),
        "Length_error": round(row["avg_abs_length_ratio_error"], 3),
        "Repetition_rate": round(row["avg_repetition_rate"], 4),
        "src_copy%": round(100 * row["source_copy_rate"], 1),
        "leak%": round(100 * row["prompt_leakage_rate"], 1),
        "latency_s": round(row["avg_latency_seconds"], 3) if pd.notna(row["avg_latency_seconds"]) else None,
    })

leaderboard_df = pd.DataFrame(lb_rows)
leaderboard_df.to_csv(LEADERBOARD_PATH, index=False, encoding="utf-8-sig")
print("LEADERBOARD (sorted by chrF++, the primary metric):")
display(leaderboard_df)

def paired_compare_against_finetuned(df: pd.DataFrame) -> pd.DataFrame:
    if "fine_tuned_llama" not in set(df["model_key"]):
        return pd.DataFrame()
    candidate_metrics = [
        "sentence_chrfpp", "sentence_bleu", "token_f1", "rouge_l_f1", "edit_similarity",
        "bertscore_f1", "comet",
    ]
    candidate_metrics = [m for m in candidate_metrics if m in df.columns and df[m].notna().any()]
    rows = []
    ft = df[df["model_key"] == "fine_tuned_llama"].set_index("row_id")
    for model_key, group in df.groupby("model_key"):
        if model_key == "fine_tuned_llama":
            continue
        other = group.set_index("row_id")
        common = sorted(set(ft.index) & set(other.index))
        if not common:
            continue
        row = {
            "comparison": "Fine-tuned Llama 3.2 1B vs " + str(other.loc[common, "model_name"].iloc[0]),
            "baseline_model_key": model_key,
            "n_paired_examples": len(common),
        }
        for metric in candidate_metrics:
            delta = ft.loc[common, metric].astype(float).to_numpy() - other.loc[common, metric].astype(float).to_numpy()
            delta = delta[~np.isnan(delta)]
            if len(delta) == 0:
                continue
            row[f"mean_delta_{metric}"] = float(delta.mean())
            row[f"median_delta_{metric}"] = float(np.median(delta))
            row[f"win_rate_{metric}"] = float((delta > 0).mean())
            row[f"tie_rate_{metric}"] = float((delta == 0).mean())
            row[f"loss_rate_{metric}"] = float((delta < 0).mean())
        rows.append(row)
    return pd.DataFrame(rows)

paired_comparison_df = paired_compare_against_finetuned(scored_predictions_df)
if len(paired_comparison_df):
    paired_comparison_df.to_csv(PAIRED_MODEL_COMPARISON_PATH, index=False, encoding="utf-8-sig")
    print("\nPAIRED COMPARISON (positive delta means fine-tuned model is better on the same examples):")
    display(paired_comparison_df)
else:
    print("\nPaired comparison skipped: fine-tuned predictions or baseline predictions are missing.")

# Auto-interpretation
if len(metrics_summary_df) >= 1:
    best = metrics_summary_df.iloc[0]
    print(f"\nBest model by chrF++: {best['model_name']} (chrF++ = {best['chrfpp']:.1f}, BLEU = {best['bleu']:.1f})")
    ft = metrics_summary_df[metrics_summary_df["model_key"] == "fine_tuned_llama"]
    base = metrics_summary_df[metrics_summary_df["model_key"] == "base_llama"]
    if len(ft) and len(base):
        d_chrf = ft.iloc[0]["chrfpp"] - base.iloc[0]["chrfpp"]
        d_bleu = ft.iloc[0]["bleu"] - base.iloc[0]["bleu"]
        print(f"Fine-tuning gain vs base Llama: chrF++ {d_chrf:+.1f}, BLEU {d_bleu:+.1f} points.")

In [ ]:
# 14. Qualitative examples -- side-by-side outputs for the report / slides

pivot = scored_predictions_df.pivot_table(
    index=["row_id", "direction", "domain", "source_text", "reference_text"],
    columns="model_name", values="prediction_text", aggfunc="first",
).reset_index()
pivot.columns.name = None

# Pick a balanced, readable set: a few per direction.
sample_parts = []
for direction, g in pivot.groupby("direction"):
    sample_parts.append(g.sample(n=min(5, len(g)), random_state=SEED))
qualitative_df = pd.concat(sample_parts, ignore_index=True)
qualitative_df.to_csv(QUALITATIVE_PATH, index=False, encoding="utf-8-sig")

pd.set_option("display.max_colwidth", 60)
print("Qualitative side-by-side examples (saved to qualitative_examples.csv):")
display(qualitative_df)

In [ ]:
# 15. External published benchmarks (CONTEXT ONLY -- different datasets, not directly comparable)

RETRIEVAL_DATE = date.today().isoformat()
external_rows = [
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "tatoeba-test-v2021-08-07", "metric": "BLEU", "value": 42.3, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "notes": "Model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "flores101-devtest", "metric": "BLEU", "value": 31.4, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "notes": "Model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "flores101-devtest", "metric": "chr-F", "value": 0.62829, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "notes": "Model-card chr-F (different scale than our chrF++)."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "tatoeba-test-v2021-08-07", "metric": "BLEU", "value": 57.6, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "notes": "Model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "flores101-devtest", "metric": "BLEU", "value": 37.6, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "notes": "Model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "flores101-devtest", "metric": "chr-F", "value": 0.64152, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "notes": "Model-card chr-F (different scale than our chrF++)."},
    {"model": "JAIST WMT17 phrase-based system", "direction": "Turkish->English", "dataset": "newstest2017", "metric": "BLEU", "value": 13.1, "source_url": "https://www.statmt.org/wmt17/pdf/WMT41.pdf", "notes": "Historical WMT17 news-domain; not comparable."},
    {"model": "JAIST WMT17 phrase-based system", "direction": "English->Turkish", "dataset": "newstest2017", "metric": "BLEU", "value": 10.4, "source_url": "https://www.statmt.org/wmt17/pdf/WMT41.pdf", "notes": "Historical WMT17 news-domain; not comparable."},
]
internet_benchmarks_df = pd.DataFrame(external_rows)
internet_benchmarks_df["retrieval_date"] = RETRIEVAL_DATE
internet_benchmarks_df.to_csv(INTERNET_BENCHMARKS_PATH, index=False, encoding="utf-8-sig")
print("Saved", INTERNET_BENCHMARKS_PATH.name, "-- context only, do NOT merge into the in-domain ranking.")
internet_benchmarks_df

In [ ]:
# 16. Plots

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

order = metrics_summary_df.sort_values("chrfpp", ascending=False)["model_name"].tolist()
model_count = max(1, len(order))

PLOT_PALETTE = {
    "bleu": "#2f6f9f",
    "chrfpp": "#2f8f5b",
    "ter": "#b65f3a",
    "bertscore_f1": "#7a5aa6",
    "comet": "#c48a2c",
}

def save_current_plot(filename: str):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()


def annotate_bars(ax, fmt="{:.1f}", min_pad=0.01):
    ymin, ymax = ax.get_ylim()
    pad = max((ymax - ymin) * min_pad, 0.01)
    for patch in ax.patches:
        height = patch.get_height()
        if pd.isna(height):
            continue
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + pad,
            fmt.format(height),
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=0,
        )

# (a1) Main translation metrics on the same 0-100-ish scale.
main_metric_cols = [c for c in ["bleu", "chrfpp", "ter"] if c in metrics_summary_df.columns]
main_metrics = metrics_summary_df.melt(
    id_vars=["model_name"], value_vars=main_metric_cols,
    var_name="metric", value_name="score"
)
plt.figure(figsize=(max(13, model_count * 2.5), 6.5))
ax = sns.barplot(
    data=main_metrics, x="model_name", y="score", hue="metric", order=order,
    palette={k: PLOT_PALETTE[k] for k in main_metric_cols}, edgecolor="0.2", linewidth=0.4,
)
upper = max(100, float(main_metrics["score"].max()) * 1.15)
ax.set_ylim(0, upper)
ax.set_title("Main Translation Metrics by Model")
ax.set_xlabel("")
ax.set_ylabel("Score (BLEU/chrF++ higher is better; TER lower is better)")
ax.tick_params(axis="x", rotation=18)
for label in ax.get_xticklabels():
    label.set_ha("right")
annotate_bars(ax, "{:.1f}")
ax.legend(title="Metric", ncol=len(main_metric_cols), frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.22))
save_current_plot("main_translation_metrics_by_model.png")

# Backward-compatible copy for reports that already reference the old filename.
plt.figure(figsize=(max(13, model_count * 2.5), 6.5))
ax = sns.barplot(
    data=main_metrics, x="model_name", y="score", hue="metric", order=order,
    palette={k: PLOT_PALETTE[k] for k in main_metric_cols}, edgecolor="0.2", linewidth=0.4,
)
ax.set_ylim(0, upper)
ax.set_title("Main Translation Metrics by Model")
ax.set_xlabel("")
ax.set_ylabel("Score (BLEU/chrF++ higher is better; TER lower is better)")
ax.tick_params(axis="x", rotation=18)
for label in ax.get_xticklabels():
    label.set_ha("right")
annotate_bars(ax, "{:.1f}")
ax.legend(title="Metric", ncol=len(main_metric_cols), frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.22))
save_current_plot("overall_metrics_by_model.png")

# (a1b) BLEU/chrF++ only. This avoids TER scale effects and is best for slides.
quality_metric_cols = [c for c in ["bleu", "chrfpp"] if c in metrics_summary_df.columns]
if quality_metric_cols:
    quality_metrics = metrics_summary_df.melt(
        id_vars=["model_name"], value_vars=quality_metric_cols,
        var_name="metric", value_name="score"
    )
    plt.figure(figsize=(max(14, model_count * 2.5), 6.2))
    ax = sns.barplot(
        data=quality_metrics, x="model_name", y="score", hue="metric", order=order,
        palette={k: PLOT_PALETTE[k] for k in quality_metric_cols}, edgecolor="0.2", linewidth=0.4,
    )
    ax.set_ylim(0, 100)
    ax.set_title("BLEU and chrF++ by Model")
    ax.set_xlabel("")
    ax.set_ylabel("Score (higher is better)")
    ax.tick_params(axis="x", rotation=20)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    annotate_bars(ax, "{:.1f}")
    ax.legend(title="Metric", ncol=len(quality_metric_cols), frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.24))
    save_current_plot("bleu_chrf_by_model.png")

# (a1c) TER separately because lower-is-better and it can exceed 100 for poor models.
if "ter" in metrics_summary_df.columns:
    ter_plot = metrics_summary_df.sort_values("ter", ascending=True).copy()
    plt.figure(figsize=(12, max(5.5, 0.55 * len(ter_plot) + 2.0)))
    ax = sns.barplot(data=ter_plot, y="model_name", x="ter", color="#b65f3a", edgecolor="0.2", linewidth=0.4)
    upper = max(50, float(ter_plot["ter"].max()) * 1.18)
    ax.set_xlim(0, upper)
    ax.set_title("TER by Model")
    ax.set_xlabel("TER (lower is better)")
    ax.set_ylabel("")
    for patch in ax.patches:
        width = patch.get_width()
        if pd.notna(width):
            ax.text(width + upper * 0.01, patch.get_y() + patch.get_height() / 2, f"{width:.1f}", va="center", fontsize=10)
    save_current_plot("ter_by_model.png")

# (a2) Neural semantic metrics use a 0-1 scale, so plot them separately.
semantic_metric_cols = [c for c in ["bertscore_f1", "comet"] if c in metrics_summary_df.columns and metrics_summary_df[c].notna().any()]
if semantic_metric_cols:
    semantic_metrics = metrics_summary_df.melt(
        id_vars=["model_name"], value_vars=semantic_metric_cols,
        var_name="metric", value_name="score"
    )
    plt.figure(figsize=(max(12, model_count * 2.4), 6.2))
    ax = sns.barplot(
        data=semantic_metrics, x="model_name", y="score", hue="metric", order=order,
        palette={k: PLOT_PALETTE.get(k, "#4c78a8") for k in semantic_metric_cols},
        edgecolor="0.2", linewidth=0.4,
    )
    low = max(0, float(semantic_metrics["score"].min()) - 0.05)
    ax.set_ylim(low, 1.0)
    ax.set_title("Semantic Similarity Metrics by Model")
    ax.set_xlabel("")
    ax.set_ylabel("Score (higher is better)")
    ax.tick_params(axis="x", rotation=18)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    annotate_bars(ax, "{:.3f}", min_pad=0.004)
    ax.legend(title="Metric", ncol=len(semantic_metric_cols), frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.22))
    save_current_plot("semantic_metrics_by_model.png")

# (b) chrF++ by direction. Use fixed 0-100 scale so direction differences are visible and comparable.
dir_plot = grouped_metrics_df[grouped_metrics_df["group_type"] == "direction"].copy()
if len(dir_plot):
    plt.figure(figsize=(12, 6.5))
    ax = sns.barplot(data=dir_plot, x="group_value", y="chrfpp", hue="model_name", hue_order=order, edgecolor="0.2", linewidth=0.4)
    ax.set_ylim(0, 100)
    ax.set_title("chrF++ by Translation Direction")
    ax.set_xlabel("Direction")
    ax.set_ylabel("chrF++ (higher is better)")
    annotate_bars(ax, "{:.0f}")
    ax.legend(title="Model", frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
    save_current_plot("direction_chrfpp_by_model.png")

# (c) App-behavior rates as percentages. Small non-zero values become readable.
behavior_cols = [c for c in ["source_copy_rate", "prompt_leakage_rate", "empty_output_rate", "extra_explanation_rate"] if c in metrics_summary_df.columns]
beh = metrics_summary_df.melt(id_vars=["model_name"], value_vars=behavior_cols, var_name="behavior", value_name="rate")
beh["percent"] = beh["rate"] * 100
plt.figure(figsize=(max(13, model_count * 2.6), 6.8))
ax = sns.barplot(data=beh, x="model_name", y="percent", hue="behavior", order=order, edgecolor="0.2", linewidth=0.4)
max_pct = float(beh["percent"].max()) if len(beh) else 0
ax.set_ylim(0, max(5, max_pct * 1.25))
ax.set_title("App-Behavior Problem Rates by Model")
ax.set_xlabel("")
ax.set_ylabel("Percent of outputs (lower is better)")
ax.tick_params(axis="x", rotation=18)
for label in ax.get_xticklabels():
    label.set_ha("right")
annotate_bars(ax, "{:.1f}%", min_pad=0.015)
ax.legend(title="Behavior", frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
save_current_plot("behavior_rates_by_model.png")

# (d) Latency. Dynamic y-axis prevents very slow models from flattening small bars too much.
if metrics_summary_df["avg_latency_seconds"].notna().any():
    latency_plot = metrics_summary_df.dropna(subset=["avg_latency_seconds"]).copy()
    plt.figure(figsize=(max(11, model_count * 2.2), 5.8))
    ax = sns.barplot(data=latency_plot, x="model_name", y="avg_latency_seconds", order=order, edgecolor="0.2", linewidth=0.4, color="#5f7f95")
    upper = max(0.2, float(latency_plot["avg_latency_seconds"].max()) * 1.25)
    ax.set_ylim(0, upper)
    ax.set_title("Average Latency per Example")
    ax.set_xlabel("")
    ax.set_ylabel("Seconds")
    ax.tick_params(axis="x", rotation=18)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    annotate_bars(ax, "{:.2f}s", min_pad=0.02)
    save_current_plot("latency_by_model.png")

# (e) Additional similarity metrics, plotted separately from BLEU/chrF/TER.
extra_metric_cols = [c for c in ["avg_token_f1", "avg_rouge_l_f1", "avg_edit_similarity"] if c in metrics_summary_df.columns]
if extra_metric_cols:
    extra_metrics = metrics_summary_df.melt(
        id_vars=["model_name"], value_vars=extra_metric_cols,
        var_name="metric", value_name="score"
    )
    plt.figure(figsize=(max(13, model_count * 2.5), 6.2))
    ax = sns.barplot(data=extra_metrics, x="model_name", y="score", hue="metric", order=order, edgecolor="0.2", linewidth=0.4)
    ax.set_ylim(0, 1.0)
    ax.set_title("Additional Similarity Metrics by Model")
    ax.set_xlabel("")
    ax.set_ylabel("Score (higher is better)")
    ax.tick_params(axis="x", rotation=18)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    annotate_bars(ax, "{:.3f}", min_pad=0.006)
    ax.legend(title="Metric", frameon=True, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=len(extra_metric_cols))
    save_current_plot("additional_similarity_metrics_by_model.png")

# (f) Per-domain chrF++ heatmaps. One overall heatmap plus split heatmaps by direction for readability.
import textwrap

def wrap_label(value, width=16):
    return "\n".join(textwrap.wrap(str(value).replace("_", " "), width=width))

def plot_domain_heatmap(data: pd.DataFrame, title: str, filename: str):
    if not len(data):
        return
    heat = data.pivot_table(index="model_name", columns="group_value", values="chrfpp").reindex(order)
    heat = heat.dropna(axis=1, how="all").dropna(axis=0, how="all")
    if heat.empty:
        return
    heat.columns = [wrap_label(c, 14) for c in heat.columns]
    width = min(max(16, 2.2 * heat.shape[1]), 34)
    height = max(6.5, 1.05 * heat.shape[0] + 3.0)
    plt.figure(figsize=(width, height))
    ax = sns.heatmap(
        heat, annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100,
        linewidths=0.6, linecolor="white", cbar_kws={"label": "chrF++"}, annot_kws={"fontsize": 10}
    )
    ax.set_title(title)
    ax.set_xlabel("Domain")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)
    save_current_plot(filename)

dom = grouped_metrics_df[grouped_metrics_df["group_type"] == "domain"].copy()
plot_domain_heatmap(dom, "chrF++ by Model and Domain", "domain_chrfpp_heatmap.png")

# Direction-specific domain heatmaps use per-example predictions, so each plot is less compressed.
if {"direction", "domain", "model_key", "model_name"}.issubset(scored_predictions_df.columns):
    direction_domain_rows = []
    for (mk, mn, direction, domain), group in scored_predictions_df.groupby(["model_key", "model_name", "direction", "domain"], sort=False):
        row = {"model_key": mk, "model_name": mn, "direction": direction, "group_value": domain}
        row.update(corpus_mt_metrics(group))
        direction_domain_rows.append(row)
    direction_domain_df = pd.DataFrame(direction_domain_rows)
    safe_names = {"English->Turkish": "english_to_turkish", "Turkish->English": "turkish_to_english"}
    for direction, part in direction_domain_df.groupby("direction"):
        filename = f"domain_chrfpp_heatmap_{safe_names.get(direction, normalize_for_match(direction).replace(' ', '_'))}.png"
        plot_domain_heatmap(part, f"chrF++ by Domain ({direction})", filename)

print("Saved readable plots to", OUTPUT_DIR)

In [ ]:
# 17. Build the Markdown evaluation report

def df_to_md(df: pd.DataFrame, max_rows: int = 30) -> str:
    return df.head(max_rows).to_markdown(index=False)

main_cols = [c for c in [
    "model_name", "n", "bleu", "chrfpp", "ter", "bertscore_f1", "comet",
    "avg_rouge_l_f1", "avg_token_f1", "avg_edit_similarity",
    "avg_length_ratio", "avg_abs_length_ratio_error", "avg_repetition_rate",
    "normalized_exact_match_rate", "empty_output_rate", "source_copy_rate",
    "prompt_leakage_rate", "extra_explanation_rate", "avg_latency_seconds",
] if c in metrics_summary_df.columns]
summary_md = metrics_summary_df[main_cols].copy()
for c in summary_md.select_dtypes(include=[float]).columns:
    summary_md[c] = summary_md[c].round(4)

dir_md = grouped_metrics_df[grouped_metrics_df["group_type"] == "direction"][
    [c for c in ["model_name", "group_value", "n", "bleu", "chrfpp", "ter", "bertscore_f1", "avg_token_f1", "avg_rouge_l_f1"] if c in grouped_metrics_df.columns]
].copy()

paired_md = paired_comparison_df.copy() if "paired_comparison_df" in globals() else pd.DataFrame()
if len(paired_md):
    keep = [c for c in [
        "comparison", "n_paired_examples",
        "mean_delta_sentence_chrfpp", "win_rate_sentence_chrfpp",
        "mean_delta_token_f1", "win_rate_token_f1",
        "mean_delta_bertscore_f1", "win_rate_bertscore_f1",
        "mean_delta_comet", "win_rate_comet",
    ] if c in paired_md.columns]
    paired_md = paired_md[keep].copy()
    for c in paired_md.select_dtypes(include=[float]).columns:
        paired_md[c] = paired_md[c].round(4)
for c in dir_md.select_dtypes(include=[float]).columns:
    dir_md[c] = dir_md[c].round(4)

best = metrics_summary_df.iloc[0]
interp = f"The best in-domain model by chrF++ is **{best['model_name']}** (chrF++ = {best['chrfpp']:.1f})."
ft = metrics_summary_df[metrics_summary_df["model_key"] == "fine_tuned_llama"]
base = metrics_summary_df[metrics_summary_df["model_key"] == "base_llama"]
if len(ft) and len(base):
    interp += (f" Fine-tuning improved chrF++ by **{ft.iloc[0]['chrfpp'] - base.iloc[0]['chrfpp']:+.1f}** "
               f"and BLEU by **{ft.iloc[0]['bleu'] - base.iloc[0]['bleu']:+.1f}** points over the base Llama-3.2-1B.")

report = f"""# Airplane Translation - Evaluation Report

Generated: {date.today().isoformat()}

## 1. Setup
The project is evaluated as a Turkish<->English airplane-domain translation system. The held-out split was
recreated from `fine_tune_chat.jsonl` with the training settings `test_size=0.02`, `seed=42`.

- Full held-out size: **{len(heldout_df)}** examples
- Evaluated in this run: **{len(eval_df)}** examples (RUN_FULL_TEST_SET = {RUN_FULL_TEST_SET})
- Models evaluated: {', '.join(metrics_summary_df['model_name'].tolist())}
- COMET model: {COMET_MODEL_ID if RUN_COMET else "disabled"}

## 2. Headline result
{interp}

### Leaderboard (sorted by chrF++)
{df_to_md(leaderboard_df)}

## 3. Metric definitions
- **BLEU** - word n-gram overlap (0-100, higher better).
- **chrF++** - character + word n-gram F-score (0-100, higher better); primary metric for morphologically rich Turkish.
- **TER** - translation edit rate (lower better).
- **BERTScore F1** - multilingual semantic similarity (0-1, higher better).
- **COMET** - learned neural quality estimate, if enabled (higher better).
- **ROUGE-L / Token F1 / Edit similarity** - additional lexical and character-level similarity checks (higher better).
- **Length ratio / length error / repetition rate** - diagnostics for too-short, too-long, or repetitive outputs.
- **Behavior rates** - source-copy / prompt-leakage / empty / extra-explanation (all lower better).

## 4. Overall in-domain metrics
{df_to_md(summary_md)}

## 5. Paired fine-tuned-vs-baseline comparison
Positive deltas mean the fine-tuned model scored higher than the baseline on the exact same examples. Win rate is the share of examples where the fine-tuned model scored higher.

{df_to_md(paired_md) if len(paired_md) else "Paired comparison was not available in this run."}

## 6. By direction
{df_to_md(dir_md, max_rows=20)}

## 7. External published benchmarks (context only)
These are from model cards / papers on **other** datasets and are **not** directly comparable to the
in-domain numbers above. They show OPUS-MT is a strong general EN<->TR system on public benchmarks.

{df_to_md(internet_benchmarks_df)}

## 8. How to use this in the report
Use the **in-domain leaderboard (Section 2)** as the primary result. The fine-tuned-vs-base comparison is
the key evidence that fine-tuning helped. Keep the external benchmarks (Section 7) in a separate
"related work / context" paragraph - never mix them into the same ranking.

## 9. Limitations
- The held-out split was the training notebook's eval split, so it is validation-style, not a fully blind test set.
- The dataset is synthetic (LLM-generated), so automatic metrics should be paired with a small human spot-check.
- Because the references come from the same project data pipeline, the fine-tuned model may receive some style advantage over general translation baselines.
- BLEU / exact-match penalize valid paraphrases; chrF++, BERTScore, and COMET reduce but do not remove this.
- External benchmarks use different tokenization, datasets, and metric settings.
- Stronger optional baselines such as NLLB-200 1.3B and MADLAD-400 3B are included as runtime toggles because they require more GPU memory and time.

## 10. Output files
`predictions.csv`, `metrics_summary.csv`, `grouped_metrics.csv`, `leaderboard.csv`,
`model_run_status.csv`, `paired_model_comparison.csv`, `qualitative_examples.csv`,
`internet_benchmarks.csv`, and the PNG charts in this folder.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print("Saved report to", REPORT_PATH)
print("\n--- Output files ---")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(" ", f.name)
print(report[:1500])

## How to read these results (quick guide for the write-up)

- **chrF++ is the headline.** It is the most reliable single number for Turkish because it works at the
  character level and is forgiving of suffix and word-order differences.
- **The most important comparison is fine-tuned vs. base Llama.** A positive gap is direct evidence that
  *your* fine-tuning produced a better airplane-domain translator than the off-the-shelf model.
- **NLLB-200 1.3B / MADLAD-400 3B are stronger general baselines.** They are included to test whether the fine-tuned model still wins against larger multilingual MT systems.
- **OPUS-MT is a strong specialist baseline.** It may match or beat the 1B models on raw BLEU/chrF++ because
  it is a dedicated translation model - that is expected and worth discussing honestly in "Results".
- **Behavior rates tell the deployment story.** Base chat models often add explanations or leak prompt text;
  a low source-copy / leakage / extra-explanation rate shows your fine-tuned model behaves like a clean
  translation API, which is the point of the app.
- **Pair the numbers with the qualitative table** (`qualitative_examples.csv`) in the slides - graders like
  to see real translations, not only metrics.

### Source notes
Public benchmark rows were taken from:
- OPUS-MT EN->TR: https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr
- OPUS-MT TR->EN: https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en
- JAIST WMT17 TR-EN system: https://www.statmt.org/wmt17/pdf/WMT41.pdf
- SacreBLEU: https://github.com/mjpost/sacrebleu - BERTScore: https://arxiv.org/abs/1904.09675 - COMET: https://github.com/Unbabel/COMET